# A2A + MCP Prototype: Weather Agent x Activity Planner Agent

This notebook builds the same system as `run_demo.py`, one layer at a time,
so you can see each piece work before it's wired into the next:

1. **MCP** - call each server's tools directly (no agent, no network).
2. **A2A** - stand the two MCP-backed services up as A2A agents and talk to
   one of them directly over HTTP.
3. **Orchestration** - the Activity Planner Agent calls the Weather Agent
   over A2A, then its own MCP tool, to answer "what should we do today?".

The Weather Agent's MCP server calls the real
[XWeather API](https://www.xweather.com/docs/weather-api) when
`XWEATHER_CLIENT_ID` / `XWEATHER_CLIENT_SECRET` are set in the environment.
Without credentials (or if a request fails) it transparently falls back to
a deterministic mock generated from `(location, date)`, so this notebook
runs offline with zero setup either way. Every weather result is tagged
`"source": "xweather"` or `"source": "mock"` so you can always tell which
one you're looking at.

Run cells top to bottom. The last cell shuts down the background agent
processes started earlier in the notebook.

In [2]:
import asyncio
import json
import sys
import time
from pathlib import Path

import httpx

# Jupyter's kernel (ipykernel) already runs its own asyncio event loop and
# supports top-level `await` in cells directly - no nest_asyncio needed
# (and it actively breaks anyio, which the MCP client relies on).

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import common.env  # loads .env (e.g. XWEATHER_CLIENT_ID/SECRET) as an import side effect
from common.mcp_client import call_tool, mcp_session
from common.a2a_protocol import A2AClient

print("Project root:", PROJECT_ROOT)

Project root: /Users/aifora/Desktop/office_laptop/learning/agents-course/mcp-course/projects/a2a-demo


## Step 1 - Call the MCP servers directly

Before any "agent" exists, each MCP server is just a subprocess exposing
tools over stdio. Let's call them the way an MCP *client* would, with no
A2A or HTTP involved yet.

In [5]:
async def demo_weather_mcp(location, days_ahead = 2):
    async with mcp_session(str(PROJECT_ROOT / "weather_agent" / "mcp_server.py")) as session:
        tools = await session.list_tools()
        print("Weather MCP tools:", [t.name for t in tools.tools])

        # "City, Country/State" disambiguates for XWeather (there are many
        # towns named "Tokyo"-adjacent or "Paris"); a bare city name still
        # works fine for the mock fallback.
        current = await call_tool(session, "get_current_weather", {"location": location})
        print(f"\nget_current_weather({location}) ->")
        print(current)

        forecast = await call_tool(session, "get_forecast", {"location": location, "days_ahead": 2})
        print(f"\nget_forecast({location}, days_ahead=2) ->")
        print(forecast)

await demo_weather_mcp(location="Paris, FR")

Weather MCP tools: ['get_current_weather', 'get_forecast']

get_current_weather(Paris, FR) ->
{
  "location": "Paris, FR",
  "date": "2026-07-09",
  "condition": "Sunny",
  "temperature_c": 31.99,
  "humidity_pct": 28,
  "wind_kph": 6.14,
  "source": "xweather"
}

get_forecast(Paris, FR, days_ahead=2) ->
{
  "location": "Paris, FR",
  "date": "2026-07-11",
  "condition": "Mostly Sunny",
  "temperature_c": 36.9,
  "humidity_pct": 58,
  "wind_kph": 9,
  "source": "xweather"
}


In [4]:
async def demo_activity_mcp():
    async with mcp_session(str(PROJECT_ROOT / "activity_agent" / "mcp_server.py")) as session:
        tools = await session.list_tools()
        print("Activity MCP tools:", [t.name for t in tools.tools])

        indoor = await call_tool(session, "suggest_indoor_activity", {})
        print("\nsuggest_indoor_activity() ->")
        print(indoor)

        recommended = await call_tool(
            session, "recommend_activity",
            {"condition": "rainy", "temperature_c": 12, "wind_kph": 20},
        )
        print("\nrecommend_activity(rainy, 12C) ->")
        print(recommended)

await demo_activity_mcp()

Activity MCP tools: ['suggest_indoor_activity', 'suggest_picnic', 'suggest_outdoor_activity', 'recommend_activity']

suggest_indoor_activity() ->
{
  "category": "indoor",
  "suggestions": [
    {
      "name": "Indoor climbing gym",
      "duration_hr": 2,
      "tip": "Rent shoes and a harness on site."
    },
    {
      "name": "Cooking or baking class",
      "duration_hr": 2,
      "tip": "Great for groups of 2-6."
    }
  ]
}

recommend_activity(rainy, 12C) ->
{
  "category": "indoor",
  "reason": "Conditions are rainy at 12.0C, best to stay indoors.",
  "suggestions": [
    {
      "name": "Cooking or baking class",
      "duration_hr": 2,
      "tip": "Great for groups of 2-6."
    },
    {
      "name": "Museum or art gallery",
      "duration_hr": 3,
      "tip": "Check for free-entry days."
    }
  ]
}


## Step 2 - Stand up the A2A agents

Each agent (`weather_agent/agent.py`, `activity_agent/agent.py`) wraps one
of the MCP servers above behind an A2A server: a FastAPI app serving an
**Agent Card** at `/.well-known/agent.json` and a `message/send` JSON-RPC
endpoint. We launch both as background subprocesses, exactly like
`run_demo.py` does, so we can talk to them over real HTTP from this
notebook.

In [6]:
def start_agent(script: str, args: list[str]):
    import subprocess
    return subprocess.Popen(
        [sys.executable, str(PROJECT_ROOT / script), *args],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )

WEATHER_URL = "http://localhost:8001"
ACTIVITY_URL = "http://localhost:8002"

weather_proc = start_agent("weather_agent/agent.py", ["--port", "8001"])
activity_proc = start_agent(
    "activity_agent/agent.py",
    ["--port", "8002", "--weather-agent-url", WEATHER_URL],
)

async def wait_until_up(url: str, timeout: float = 20.0):
    deadline = time.monotonic() + timeout
    async with httpx.AsyncClient() as client:
        while time.monotonic() < deadline:
            try:
                resp = await client.get(f"{url}/.well-known/agent.json", timeout=2.0)
                if resp.status_code == 200:
                    return
            except httpx.HTTPError:
                pass
            await asyncio.sleep(0.5)
    raise TimeoutError(f"{url} did not come up in time")

await wait_until_up(WEATHER_URL)
await wait_until_up(ACTIVITY_URL)
print("Both agents are up.")

Both agents are up.


## Step 3 - Discover agents via their Agent Card

This is the A2A analogue of MCP's `tools/list`: instead of listing tools,
you fetch a small JSON document describing who the agent is and what it can
do, *before* sending it any messages.

In [7]:
weather_client = A2AClient(WEATHER_URL)
activity_client = A2AClient(ACTIVITY_URL)

weather_card = await weather_client.get_agent_card()
activity_card = await activity_client.get_agent_card()

for card in (weather_card, activity_card):
    print(f"{card.name} @ {card.url}")
    print(f"  {card.description}")
    for skill in card.skills:
        print(f"  - skill '{skill.id}': {skill.description}")
    print()

Weather Agent @ http://localhost:8001
  Reports current weather and short-range forecasts for a city.
  - skill 'get_weather': Get current or forecast weather for a location, e.g. 'weather in Paris' or 'forecast for Tokyo tomorrow'.

Activity Planner Agent @ http://localhost:8002
  Plans indoor, picnic, or outdoor activities based on the weather in a given city. Delegates weather lookups to the Weather Agent over A2A.
  - skill 'plan_activity': Suggest an activity for a city, weather-aware, e.g. 'what should we do in Paris today?'



## Step 4 - Talk to a single agent over A2A

A direct `message/send` call to the Weather Agent. Under the hood this is a
JSON-RPC POST to `http://localhost:8001/`, and the Weather Agent answers it
by calling its own MCP server internally.

In [8]:
reply = await weather_client.send_message("What's the weather in Nairobi, KE tomorrow?")
print(reply)

Weather for Nairobi, KE on 2026-07-10: Mostly Cloudy with Showers, 21.6C, humidity 95%, wind 2.0 kph. (source: xweather)


## Step 5 - Watch agent-to-agent orchestration

Now the interesting part. The Activity Planner Agent doesn't have any
weather logic of its own - when we ask it to plan something, it:

1. Extracts the city from our message.
2. Sends its **own** A2A `message/send` request to the Weather Agent.
3. Feeds the weather back into its **own** MCP `recommend_activity` tool.
4. Replies to us with a weather-aware suggestion.

From here, it's indistinguishable from calling any other agent - the fact
that it fans out to a second agent and a local MCP server is an
implementation detail.

In [9]:
queries = [
    "What should we do in Paris, FR today?",
    "Plan something fun in New York, NY.",
    "What can we do in Tokyo, JP today?",
]

for q in queries:
    reply = await activity_client.send_message(q)
    print(f"USER: {q}")
    print(f"ACTIVITY PLANNER AGENT: {reply}\n")

USER: What should we do in Paris, FR today?
ACTIVITY PLANNER AGENT: For Paris, FR (Sunny, 32.0C, humidity 28%, wind 6.2 kph): Sunny weather at 32.0C is good for an active outing. Suggestions -> City walking tour (~2h, tip: Wear comfortable shoes.); Kayaking (~2h, tip: Life jackets are usually available for rent.) [weather data: xweather]

USER: Plan something fun in New York, NY.
ACTIVITY PLANNER AGENT: For New York, NY (Mostly Sunny, 28.3C, humidity 71%, wind 7.2 kph): Mild Mostly Sunny weather at 28.3C with low wind, perfect for a picnic. Suggestions -> Riverside park (~2h, tip: Bring a blanket, arrive before noon for shade.); Botanical garden lawn (~2h, tip: Check garden opening hours.) [weather data: xweather]

USER: What can we do in Tokyo, JP today?
ACTIVITY PLANNER AGENT: For Tokyo, JP (Mostly Clear, 24.7C, humidity 87%, wind 4.8 kph): Mild Mostly Clear weather at 24.7C with low wind, perfect for a picnic. Suggestions -> Riverside park (~2h, tip: Bring a blanket, arrive before n

In [10]:
!which python

/Users/aifora/Desktop/office_laptop/work/ai-agent-service/.venv/bin/python


## Step 6 - Shut down

Stop the background agent processes started in Step 2.

In [11]:
await weather_client.aclose()
await activity_client.aclose()

for proc in (weather_proc, activity_proc):
    proc.terminate()
for proc in (weather_proc, activity_proc):
    try:
        proc.wait(timeout=5)
    except Exception:
        proc.kill()

print("Agents stopped.")

Agents stopped.


## Architecture

```mermaid
flowchart TB
    User(["User / Client"])

    subgraph AA["Activity Planner Agent  (:8002)"]
        direction TB
        AA_A2A["A2A server\nAgent Card + message/send"]
        AA_Client["A2A client\n(calls Weather Agent)"]
        AA_MCPClient["MCP client (stdio)"]
        AA_A2A --> AA_Client
        AA_A2A --> AA_MCPClient
    end

    subgraph AA_MCP["Activity MCP Server (subprocess)"]
        direction TB
        T1["suggest_indoor_activity"]
        T2["suggest_picnic"]
        T3["suggest_outdoor_activity"]
        T4["recommend_activity"]
    end

    subgraph WA["Weather Agent  (:8001)"]
        direction TB
        WA_A2A["A2A server\nAgent Card + message/send"]
        WA_MCPClient["MCP client (stdio)"]
        WA_A2A --> WA_MCPClient
    end

    subgraph WA_MCP["Weather MCP Server (subprocess)"]
        direction TB
        W1["get_current_weather"]
        W2["get_forecast"]
    end

    User -- "A2A: message/send\n'what should we do in Paris today?'" --> AA_A2A
    AA_Client -- "A2A: message/send\n'weather in Paris?'" --> WA_A2A
    WA_A2A -- "A2A: Task(completed)\n'sunny, 22C...'" --> AA_Client
    AA_MCPClient -- "MCP: tools/call\nrecommend_activity(condition, temp)" --> AA_MCP
    AA_MCP -- "MCP: tool result" --> AA_MCPClient
    WA_MCPClient -- "MCP: tools/call\nget_current_weather(location)" --> WA_MCP
    WA_MCP -- "MCP: tool result" --> WA_MCPClient
    AA_A2A -- "A2A: Task(completed)\nweather-aware activity plan" --> User
```

**Two protocols, two jobs:**
- **A2A** (Activity Planner <-> Weather Agent) - peer agents, each with its
  own Agent Card, discoverable and callable over HTTP/JSON-RPC.
- **MCP** (each agent <-> its own subprocess server) - an agent's private
  toolbox, not exposed to other agents.

See `../README.md` for the full write-up.